# TAIRO — Cosmos World-Model + Gymnasium RL + Robot Attack Integration

This notebook provides code examples for the Week 7 improvements:

1. Keep the existing SAC+HER / Gymnasium PickAndPlace attack benchmark.
2. Add explicit **Cosmos/world-model integration**.
3. Log Gymnasium rollouts into a format usable by a world-model module.
4. Call Cosmos through a local NVIDIA NIM-style HTTP API or through local Cosmos scripts.
5. Use Cosmos as a **recovery evaluator**, not necessarily as a full policy replacement.
6. Compare clean, attacked, randomized, and recovered policies.

The notebook is designed to run in three modes:

- **Demo mode**: no MuJoCo, no Gymnasium Robotics, no Cosmos required.
- **Gymnasium mode**: uses `gymnasium-robotics` FetchPickAndPlace if installed.
- **Cosmos mode**: calls a local or hosted Cosmos endpoint if configured.

The main goal is to help the team show a concrete Cosmos milestone next week, even if full Cosmos training is not complete.

## Installation notes

Run these in a terminal or Colab cell as needed.

```bash
pip install gymnasium gymnasium-robotics stable-baselines3 imageio opencv-python requests pandas scikit-learn matplotlib
```

For MuJoCo/Gymnasium Robotics, local installation can require additional system packages.  
For Cosmos, use one of these options:

1. **Local NVIDIA NIM endpoint** exposing `/v1/infer`.
2. **Hosted NVIDIA API Catalog endpoint** copied into `NVIDIA_COSMOS_INVOKE_URL`.
3. **Local Cosmos repo scripts**, such as `examples/video2world.py`.

Do not hardcode API keys in the notebook. Use environment variables.

In [ ]:
# =========================
# 0. Imports and config
# =========================

from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Dict, Any, List, Optional, Tuple
import os
import io
import json
import time
import base64
import subprocess
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import requests
except Exception:
    requests = None

try:
    import imageio.v2 as imageio
except Exception:
    imageio = None

try:
    from PIL import Image
except Exception:
    Image = None

try:
    import cv2
except Exception:
    cv2 = None

try:
    import gymnasium as gym
except Exception:
    gym = None

try:
    import gymnasium_robotics
except Exception:
    gymnasium_robotics = None

try:
    from stable_baselines3 import SAC
    from stable_baselines3.her.her_replay_buffer import HerReplayBuffer
except Exception:
    SAC = None
    HerReplayBuffer = None

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

PROJECT_DIR = Path("tairo_cosmos_project")
DATA_DIR = PROJECT_DIR / "data"
FRAME_DIR = PROJECT_DIR / "frames"
VIDEO_DIR = PROJECT_DIR / "videos"
OUTPUT_DIR = PROJECT_DIR / "outputs"

for p in [DATA_DIR, FRAME_DIR, VIDEO_DIR, OUTPUT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# Cosmos endpoint configuration.
# Local NIM example: http://0.0.0.0:8000
COSMOS_BASE_URL = os.environ.get("COSMOS_BASE_URL", "http://0.0.0.0:8000")
NVIDIA_API_KEY = os.environ.get("NVIDIA_API_KEY", "")
NVIDIA_COSMOS_INVOKE_URL = os.environ.get("NVIDIA_COSMOS_INVOKE_URL", "")

# Keep remote/API execution off by default.
RUN_GYMNASIUM = False
RUN_COSMOS_API = False
RUN_COSMOS_LOCAL_SCRIPT = False

print("Gymnasium installed:", gym is not None)
print("Gymnasium Robotics installed:", gymnasium_robotics is not None)
print("Stable-Baselines3 SAC installed:", SAC is not None)
print("Requests installed:", requests is not None)

## Core idea

The team already has SAC+HER policies and attacks. The missing part is a concrete Cosmos/world-model step.

A realistic Week 7 integration can be:

1. Roll out the SAC+HER policy in `FetchPickAndPlace`.
2. Apply attacks such as object position spoofing, contact dropout, and grip-state falsification.
3. Save state/action/frame logs.
4. When a failure is detected, generate candidate recovery actions.
5. Use Cosmos/world-model prediction to evaluate candidate futures.
6. Choose the action with the best predicted recovery score.
7. Compare: clean policy, attacked policy, randomized policy, and world-model recovery.

In [ ]:
# ==================================================
# 1. FetchPickAndPlace observation index configuration
# ==================================================
# Gymnasium Robotics Fetch observations are usually a dict:
# {
#   "observation": np.array([...]),
#   "achieved_goal": np.array([...]),
#   "desired_goal": np.array([...])
# }
#
# The exact observation vector layout should be verified in your environment.
# For classic FetchPickAndPlace, object position is commonly observation[3:6].
# Always print one observation before relying on these indices.

FETCH_PICKPLACE_INDICES = {
    "gripper_pos": slice(0, 3),
    "object_pos": slice(3, 6),
    "object_rel_pos": slice(6, 9),
    "gripper_state": slice(9, 11),
    "object_rot": slice(11, 14),
    "object_velp": slice(14, 17),
    "object_velr": slice(17, 20),
    "gripper_velp": slice(20, 23),
    "gripper_vel": slice(23, 25),
}

def describe_fetch_observation(obs):
    if isinstance(obs, dict):
        arr = obs.get("observation", None)
        print("Observation dict keys:", list(obs.keys()))
        if arr is not None:
            print("observation shape:", np.asarray(arr).shape)
            for name, sl in FETCH_PICKPLACE_INDICES.items():
                try:
                    print(f"{name:16s}", np.asarray(arr)[sl])
                except Exception:
                    pass
        if "achieved_goal" in obs:
            print("achieved_goal:", obs["achieved_goal"])
        if "desired_goal" in obs:
            print("desired_goal:", obs["desired_goal"])
    else:
        print("Observation type:", type(obs))
        print("Observation shape:", np.asarray(obs).shape)

# Example:
# env = gym.make("FetchPickAndPlace-v2", render_mode="rgb_array")
# obs, info = env.reset()
# describe_fetch_observation(obs)

In [ ]:
# ======================================================
# 2. Attack wrappers for Gymnasium / Gymnasium Robotics
# ======================================================

class ActionAttackWrapper(gym.ActionWrapper if gym is not None else object):
    '''
    Applies action-space attacks:
    - action_clipping
    - action_delay
    - action_reversal
    - grip_state_falsification
    '''
    def __init__(self, env, attack_type="none", severity=1.0, delay_steps=2):
        if gym is not None:
            super().__init__(env)
        else:
            self.env = env

        self.attack_type = attack_type
        self.severity = severity
        self.delay_steps = delay_steps
        self.action_buffer = []

    def action(self, action):
        a = np.asarray(action, dtype=np.float32).copy()

        if self.attack_type == "none":
            return a

        if self.attack_type == "action_clipping":
            # Limit control authority.
            clip = max(0.05, 1.0 - 0.7 * self.severity)
            return np.clip(a, -clip, clip)

        if self.attack_type == "action_delay":
            self.action_buffer.append(a.copy())
            if len(self.action_buffer) <= self.delay_steps:
                return np.zeros_like(a)
            return self.action_buffer.pop(0)

        if self.attack_type == "action_reversal":
            return -a

        if self.attack_type == "grip_state_falsification":
            # Fetch action often has 4 dims: dx, dy, dz, gripper.
            # Invert only gripper channel.
            if len(a) >= 4:
                a[3] = -a[3]
            return a

        return a

class ObservationAttackWrapper(gym.ObservationWrapper if gym is not None else object):
    '''
    Applies observation-space attacks to Fetch-style dict observations.
    '''
    def __init__(self, env, attack_type="none", severity=1.0, indices=None, rng_seed=42):
        if gym is not None:
            super().__init__(env)
        else:
            self.env = env

        self.attack_type = attack_type
        self.severity = severity
        self.indices = indices or FETCH_PICKPLACE_INDICES
        self.rng = np.random.default_rng(rng_seed)

    def observation(self, obs):
        if self.attack_type == "none":
            return obs

        if not isinstance(obs, dict) or "observation" not in obs:
            return obs

        attacked = {k: np.array(v).copy() if isinstance(v, np.ndarray) else v for k, v in obs.items()}
        arr = attacked["observation"].copy()

        if self.attack_type == "object_position_spoof":
            offset = self.rng.uniform(-0.10, 0.10, size=3) * self.severity
            arr[self.indices["object_pos"]] += offset
            if "achieved_goal" in attacked:
                attacked["achieved_goal"] = attacked["achieved_goal"] + offset

        elif self.attack_type == "sensor_bias":
            bias = 0.05 * self.severity
            arr += bias

        elif self.attack_type == "sensor_dropout":
            mask = self.rng.random(arr.shape) < min(0.8, 0.25 * self.severity)
            arr[mask] = 0.0

        elif self.attack_type == "contact_dropout":
            # Approximation of "camera died, gripper preserved":
            # zero object pose and velocity channels while preserving gripper state.
            for key in ["object_pos", "object_rel_pos", "object_rot", "object_velp", "object_velr"]:
                arr[self.indices[key]] = 0.0
            if "achieved_goal" in attacked:
                attacked["achieved_goal"] = np.zeros_like(attacked["achieved_goal"])

        attacked["observation"] = arr
        return attacked

def make_pickplace_env(
    render_mode="rgb_array",
    obs_attack="none",
    action_attack="none",
    severity=1.0,
    delay_steps=2,
):
    if gym is None:
        raise ImportError("gymnasium is not installed.")

    if gymnasium_robotics is None:
        raise ImportError("gymnasium_robotics is not installed.")

    env = gym.make("FetchPickAndPlace-v2", render_mode=render_mode)
    env = ObservationAttackWrapper(env, attack_type=obs_attack, severity=severity)
    env = ActionAttackWrapper(env, attack_type=action_attack, severity=severity, delay_steps=delay_steps)
    return env

In [ ]:
# =======================================
# 3. Random attack scheduler wrapper
# =======================================
# This implements a safer version of the team's "coin flip" randomized attack training.
# Instead of attacking too often from the start, use a curriculum.

class RandomAttackScheduler:
    def __init__(
        self,
        clean_prob=0.8,
        attack_types=None,
        severity_schedule=None,
        rng_seed=42
    ):
        self.clean_prob = clean_prob
        self.attack_types = attack_types or [
            "object_position_spoof",
            "contact_dropout",
            "sensor_bias",
            "sensor_dropout",
            "action_clipping",
            "action_delay",
            "action_reversal",
            "grip_state_falsification",
        ]
        self.severity_schedule = severity_schedule or (lambda episode_idx: 0.25)
        self.rng = np.random.default_rng(rng_seed)

    def sample_attack(self, episode_idx=0):
        if self.rng.random() < self.clean_prob:
            return {"obs_attack": "none", "action_attack": "none", "severity": 0.0}

        attack = self.rng.choice(self.attack_types)
        severity = float(self.severity_schedule(episode_idx))

        if attack in {"object_position_spoof", "contact_dropout", "sensor_bias", "sensor_dropout"}:
            return {"obs_attack": attack, "action_attack": "none", "severity": severity}

        return {"obs_attack": "none", "action_attack": attack, "severity": severity}

def linear_curriculum(start=0.1, end=1.0, total_episodes=1000):
    def schedule(ep):
        alpha = min(1.0, ep / max(1, total_episodes))
        return start + alpha * (end - start)
    return schedule

# Recommended ablations:
schedulers = {
    "mostly_clean": RandomAttackScheduler(clean_prob=0.8, severity_schedule=linear_curriculum(0.1, 0.5, 1000)),
    "balanced": RandomAttackScheduler(clean_prob=0.5, severity_schedule=linear_curriculum(0.1, 0.7, 1000)),
    "aggressive": RandomAttackScheduler(clean_prob=0.2, severity_schedule=linear_curriculum(0.2, 1.0, 1000)),
}

In [ ]:
# ==========================================
# 4. Policy interface and demo random policy
# ==========================================

class RandomPolicy:
    def __init__(self, action_space=None, action_dim=4):
        self.action_space = action_space
        self.action_dim = action_dim

    def predict(self, obs, deterministic=True):
        if self.action_space is not None:
            return self.action_space.sample(), None
        return np.random.uniform(-1, 1, size=(self.action_dim,)).astype(np.float32), None

def load_sac_model(model_path):
    if SAC is None:
        raise ImportError("stable-baselines3 is not installed.")
    return SAC.load(model_path)

# Example:
# clean_policy = load_sac_model("models/sac_her_pickandplace_clean_2m.zip")
# randomized_policy = load_sac_model("models/sac_her_pickandplace_randomized_2m.zip")

In [ ]:
# ========================================
# 5. Rollout logging with frames and states
# ========================================

@dataclass
class StepLog:
    episode_id: int
    t: int
    condition: str
    obs_attack: str
    action_attack: str
    severity: float
    action: List[float]
    reward: float
    terminated: bool
    truncated: bool
    success: float
    final_distance: float
    object_displacement: float
    grasp_proxy: float
    frame_path: Optional[str]

def get_goal_distance(obs):
    if isinstance(obs, dict) and "achieved_goal" in obs and "desired_goal" in obs:
        return float(np.linalg.norm(np.asarray(obs["achieved_goal"]) - np.asarray(obs["desired_goal"])))
    return np.nan

def get_object_pos(obs):
    if isinstance(obs, dict) and "observation" in obs:
        arr = np.asarray(obs["observation"])
        return arr[FETCH_PICKPLACE_INDICES["object_pos"]].copy()
    return np.array([np.nan, np.nan, np.nan])

def grasp_proxy_from_obs(obs):
    '''
    Lightweight proxy. In real experiments, replace this with
    simulator contacts or gripper-object distance.
    '''
    if isinstance(obs, dict) and "observation" in obs:
        arr = np.asarray(obs["observation"])
        object_rel = arr[FETCH_PICKPLACE_INDICES["object_rel_pos"]]
        return float(np.linalg.norm(object_rel) < 0.05)
    return np.nan

def save_frame(env, episode_id, t, condition):
    if imageio is None:
        return None
    try:
        frame = env.render()
        if frame is None:
            return None
        path = FRAME_DIR / f"{condition}_ep{episode_id:04d}_t{t:03d}.png"
        imageio.imwrite(path, frame)
        return str(path)
    except Exception:
        return None

def rollout_episode(
    env,
    policy,
    episode_id=0,
    condition="clean",
    obs_attack="none",
    action_attack="none",
    severity=0.0,
    max_steps=150,
    save_frames=True,
):
    obs, info = env.reset()
    initial_object_pos = get_object_pos(obs)
    logs = []

    for t in range(max_steps):
        action, _ = policy.predict(obs, deterministic=True)

        next_obs, reward, terminated, truncated, info = env.step(action)

        success = float(info.get("is_success", 0.0)) if isinstance(info, dict) else 0.0
        final_distance = get_goal_distance(next_obs)
        object_displacement = float(np.linalg.norm(get_object_pos(next_obs) - initial_object_pos))
        grasp_proxy = grasp_proxy_from_obs(next_obs)

        frame_path = save_frame(env, episode_id, t, condition) if save_frames else None

        logs.append(StepLog(
            episode_id=episode_id,
            t=t,
            condition=condition,
            obs_attack=obs_attack,
            action_attack=action_attack,
            severity=severity,
            action=np.asarray(action).tolist(),
            reward=float(reward),
            terminated=bool(terminated),
            truncated=bool(truncated),
            success=success,
            final_distance=final_distance,
            object_displacement=object_displacement,
            grasp_proxy=grasp_proxy,
            frame_path=frame_path,
        ))

        obs = next_obs

        if terminated or truncated:
            break

    return pd.DataFrame([asdict(x) for x in logs])

# Demo mode creates fake logs so the rest of the notebook can run.
def make_demo_rollout_logs(n_episodes=30, max_steps=50):
    rows = []
    rng = np.random.default_rng(RANDOM_STATE)
    conditions = [
        ("clean", "none", "none", 0.0, 0.95),
        ("object_position_spoof", "object_position_spoof", "none", 0.7, 0.20),
        ("contact_dropout", "contact_dropout", "none", 0.7, 0.05),
        ("grip_state_falsification", "none", "grip_state_falsification", 0.7, 0.00),
        ("action_delay", "none", "action_delay", 0.7, 0.75),
        ("action_clipping", "none", "action_clipping", 0.7, 0.85),
    ]

    ep = 0
    for condition, obs_attack, action_attack, severity, base_success in conditions:
        for _ in range(n_episodes):
            success = rng.random() < base_success
            final_distance = rng.normal(0.03 if success else 0.33, 0.02)
            displacement = rng.normal(0.25 if success else 0.02, 0.03)
            for t in range(max_steps):
                frac = (t + 1) / max_steps
                rows.append({
                    "episode_id": ep,
                    "t": t,
                    "condition": condition,
                    "obs_attack": obs_attack,
                    "action_attack": action_attack,
                    "severity": severity,
                    "action": rng.uniform(-1, 1, size=4).tolist(),
                    "reward": float(success) - 1.0,
                    "terminated": t == max_steps - 1,
                    "truncated": False,
                    "success": float(success and t == max_steps - 1),
                    "final_distance": max(0, final_distance + (1-frac)*0.2),
                    "object_displacement": max(0, displacement * frac),
                    "grasp_proxy": float(displacement > 0.1 and t > max_steps/3),
                    "frame_path": None,
                })
            ep += 1
    return pd.DataFrame(rows)

if RUN_GYMNASIUM and gym is not None and gymnasium_robotics is not None:
    env = make_pickplace_env()
    policy = RandomPolicy(env.action_space)
    logs_df = rollout_episode(env, policy, episode_id=0, condition="clean")
else:
    logs_df = make_demo_rollout_logs()

logs_df.head()

In [ ]:
# =====================================================
# 6. Benchmark table: clean vs attacked vs randomized
# =====================================================

def summarize_episode_logs(logs_df):
    final = (
        logs_df.sort_values(["episode_id", "t"])
        .groupby("episode_id")
        .tail(1)
        .copy()
    )

    summary = (
        final
        .groupby(["condition", "obs_attack", "action_attack"], as_index=False)
        .agg(
            episodes=("episode_id", "count"),
            success_rate=("success", "mean"),
            mean_final_distance=("final_distance", "mean"),
            median_final_distance=("final_distance", "median"),
            mean_object_displacement=("object_displacement", "mean"),
            mean_grasp_proxy=("grasp_proxy", "mean"),
        )
        .sort_values("success_rate", ascending=False)
    )
    return summary, final

benchmark_table, final_logs = summarize_episode_logs(logs_df)
benchmark_table.round(3)

In [ ]:
plt.figure(figsize=(10, 4))
plot_df = benchmark_table.sort_values("success_rate", ascending=False)
plt.bar(plot_df["condition"], plot_df["success_rate"])
plt.ylabel("Success rate")
plt.xlabel("Condition")
plt.title("PickAndPlace success under clean and attack conditions")
plt.xticks(rotation=35, ha="right")
plt.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ===============================================
# 7. Failure-mode labels for classifier/recovery
# ===============================================

def label_failure_mode(row):
    if row["success"] >= 0.5:
        return "success"

    if row["object_displacement"] < 0.05 and row["grasp_proxy"] < 0.5:
        return "never_grasped_object"

    if row["grasp_proxy"] >= 0.5 and row["object_displacement"] < 0.10:
        return "grasped_but_no_transport"

    if row["object_displacement"] >= 0.10 and row["final_distance"] > 0.12:
        return "moved_object_wrong_direction"

    if row["obs_attack"] in {"object_position_spoof", "sensor_bias"}:
        return "perception_spoof_failure"

    if row["obs_attack"] in {"contact_dropout", "sensor_dropout"}:
        return "sensor_dropout_failure"

    if row["action_attack"] in {"grip_state_falsification", "action_reversal"}:
        return "control_channel_failure"

    return "unknown_failure"

final_logs["failure_mode"] = final_logs.apply(label_failure_mode, axis=1)
final_logs[["episode_id", "condition", "success", "final_distance", "object_displacement", "grasp_proxy", "failure_mode"]].head()

In [ ]:
failure_table = (
    final_logs
    .groupby(["condition", "failure_mode"], as_index=False)
    .agg(episodes=("episode_id", "count"))
    .sort_values(["condition", "episodes"], ascending=[True, False])
)

failure_table

In [ ]:
# Train a simple failure-mode classifier from rollout summaries.
# In real experiments, add features such as state deltas, action norms, distance derivatives, and attack flags.

classifier_df = final_logs.copy()

classifier_df["obs_attack_code"] = classifier_df["obs_attack"].astype("category").cat.codes
classifier_df["action_attack_code"] = classifier_df["action_attack"].astype("category").cat.codes

feature_cols = [
    "final_distance",
    "object_displacement",
    "grasp_proxy",
    "severity",
    "obs_attack_code",
    "action_attack_code",
]

X = classifier_df[feature_cols].fillna(0)
y = classifier_df["failure_mode"]

if len(y.unique()) > 1 and len(classifier_df) > 20:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
    )
    clf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, class_weight="balanced")
    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)
    print(classification_report(y_test, pred, zero_division=0))
else:
    clf = None
    print("Not enough class diversity for classifier.")

# 8. Cosmos API integration

There are two practical paths:

1. **NVIDIA NIM-style local endpoint**  
   A Cosmos NIM endpoint usually exposes health and inference routes such as `/v1/health/ready` and `/v1/infer`.

2. **Hosted NVIDIA API Catalog**  
   Hosted endpoints use an invoke URL and API key. Copy the model-specific invoke URL into `NVIDIA_COSMOS_INVOKE_URL` and set `NVIDIA_API_KEY`.

For Week 7, the safest milestone is:

- Capture a frame from Gymnasium.
- Build a physical prompt describing the current robot state and attack.
- Call Cosmos to generate a predicted future video or use the local script.
- Score the predicted future with a simple heuristic.
- Select a recovery action.

Even if full Cosmos action-conditioned training is not ready, this demonstrates a real integration path.

In [ ]:
# ======================================
# 8A. Utilities: images, videos, base64
# ======================================

def image_to_base64(image_path):
    image_path = Path(image_path)
    with image_path.open("rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

def base64_to_file(b64_string, output_path):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("wb") as f:
        f.write(base64.b64decode(b64_string))
    return output_path

def frame_array_to_png(frame, output_path):
    if imageio is None:
        raise ImportError("imageio is required to save frames.")
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    imageio.imwrite(output_path, frame)
    return output_path

def make_video_from_frames(frame_paths, output_path, fps=10):
    if imageio is None:
        raise ImportError("imageio is required to write videos.")
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    frames = [imageio.imread(str(p)) for p in frame_paths]
    imageio.mimsave(str(output_path), frames, fps=fps)
    return output_path

In [ ]:
# ============================================
# 8B. Cosmos HTTP client: local NIM / hosted API
# ============================================

class CosmosClient:
    '''
    Minimal HTTP client for Cosmos-like endpoints.

    Supports:
    - local NIM: BASE_URL/v1/health/ready and BASE_URL/v1/infer
    - hosted API catalog: NVIDIA_COSMOS_INVOKE_URL with Bearer auth

    The NIM text/video generation response may include b64_video.
    For model-specific schemas, inspect /openapi.json or /v1/metadata.
    '''
    def __init__(
        self,
        base_url=COSMOS_BASE_URL,
        api_key=NVIDIA_API_KEY,
        invoke_url=NVIDIA_COSMOS_INVOKE_URL,
        timeout=600,
    ):
        if requests is None:
            raise ImportError("requests is required for Cosmos API calls.")

        self.base_url = base_url.rstrip("/")
        self.api_key = api_key
        self.invoke_url = invoke_url
        self.timeout = timeout

    def headers(self):
        h = {
            "Accept": "application/json",
            "Content-Type": "application/json",
        }
        if self.api_key:
            h["Authorization"] = f"Bearer {self.api_key}"
        return h

    def health_ready(self):
        url = f"{self.base_url}/v1/health/ready"
        r = requests.get(url, headers=self.headers(), timeout=30)
        r.raise_for_status()
        return r.json()

    def metadata(self):
        url = f"{self.base_url}/v1/metadata"
        r = requests.get(url, headers=self.headers(), timeout=30)
        r.raise_for_status()
        return r.json()

    def openapi(self):
        url = f"{self.base_url}/openapi.json"
        r = requests.get(url, headers=self.headers(), timeout=30)
        if r.status_code == 404:
            return None
        r.raise_for_status()
        return r.json()

    def infer_local_nim(self, payload: Dict[str, Any]):
        url = f"{self.base_url}/v1/infer"
        r = requests.post(url, headers=self.headers(), json=payload, timeout=self.timeout)
        r.raise_for_status()
        return r.json()

    def infer_hosted(self, payload: Dict[str, Any]):
        if not self.invoke_url:
            raise ValueError("Set NVIDIA_COSMOS_INVOKE_URL for hosted API calls.")
        r = requests.post(self.invoke_url, headers=self.headers(), json=payload, timeout=self.timeout)
        r.raise_for_status()
        return r.json()

def build_cosmos_text2world_payload(
    prompt,
    negative_prompt="blurry, low quality, artifacts, unsafe motion, physically impossible motion",
    seed=4,
    height=704,
    width=1280,
    frames_count=121,
    fps=24,
    steps=50,
    guidance_scale=7.5,
):
    '''
    Payload pattern for a local Cosmos NIM /v1/infer Text2World-style request.
    Keep resolution/frame count small if your endpoint supports smaller profiles.
    '''
    return {
        "prompt": prompt,
        "negative_prompt": negative_prompt,
        "prompt_upsampling": True,
        "seed": seed,
        "guidance_scale": guidance_scale,
        "steps": steps,
        "video_params": {
            "height": height,
            "width": width,
            "frames_count": frames_count,
            "frames_per_sec": fps,
        },
    }

def cosmos_generate_video_from_prompt(prompt, output_path, client=None):
    '''
    Concrete NIM-style API call.
    RUN_COSMOS_API must be True to actually call the endpoint.
    '''
    if client is None:
        client = CosmosClient()

    payload = build_cosmos_text2world_payload(prompt)

    if not RUN_COSMOS_API:
        print("RUN_COSMOS_API=False. Returning placeholder path.")
        return None

    result = client.infer_local_nim(payload)

    if "b64_video" in result:
        return base64_to_file(result["b64_video"], output_path)

    # Some hosted endpoints may return a URL or async request id.
    print("Cosmos response keys:", list(result.keys()))
    return result

In [ ]:
# ========================================================
# 8C. Cosmos local script integration for Video2World
# ========================================================
# Use this path when the team has cloned the Cosmos repo locally.
# This is often easier for image/video-conditioned prediction than a generic API endpoint.

def run_cosmos_video2world_script(
    cosmos_repo_dir,
    input_path,
    prompt,
    output_video,
    model_size="2B",
    num_conditional_frames=1,
    extra_args=None,
):
    '''
    Calls:
        python -m examples.video2world --model_size 2B --input_path ... --prompt ... --save_path ...

    Requirements:
    - Cosmos repo installed and checkpoints downloaded.
    - Run from the Cosmos repo directory.
    '''
    cosmos_repo_dir = Path(cosmos_repo_dir)
    input_path = Path(input_path)
    output_video = Path(output_video)
    output_video.parent.mkdir(parents=True, exist_ok=True)

    cmd = [
        "python", "-m", "examples.video2world",
        "--model_size", model_size,
        "--input_path", str(input_path),
        "--num_conditional_frames", str(num_conditional_frames),
        "--prompt", prompt,
        "--save_path", str(output_video),
    ]

    if extra_args:
        cmd.extend(extra_args)

    if not RUN_COSMOS_LOCAL_SCRIPT:
        print("RUN_COSMOS_LOCAL_SCRIPT=False. Command would be:")
        print(" ".join(cmd))
        return output_video

    result = subprocess.run(
        cmd,
        cwd=str(cosmos_repo_dir),
        capture_output=True,
        text=True,
        check=False,
    )

    print("stdout:", result.stdout[-1000:])
    print("stderr:", result.stderr[-1000:])

    if result.returncode != 0:
        raise RuntimeError(f"Cosmos script failed with code {result.returncode}")

    return output_video

# Example:
# run_cosmos_video2world_script(
#     cosmos_repo_dir="/path/to/cosmos",
#     input_path="tairo_cosmos_project/frames/current_state.png",
#     prompt="A robot arm recovers from a failed grasp and moves the black cube toward the red goal marker.",
#     output_video="tairo_cosmos_project/videos/cosmos_recovery_prediction.mp4",
# )

In [ ]:
# =========================================================
# 8D. Action-conditioned Cosmos adapter placeholder
# =========================================================
# Cosmos Predict2.5 includes a robot/action-conditioned model option in current docs.
# The exact deployed endpoint schema can vary. This adapter keeps the team's code clean:
# swap payload fields after checking /openapi.json or the local training script.

def build_action_conditioned_payload(
    current_frame_path,
    action_sequence,
    prompt,
    seed=4,
    fps=10,
):
    '''
    Generic action-conditioned payload template.

    current_frame_path: PNG/JPG frame from Gymnasium render.
    action_sequence: list of action vectors, e.g., [[dx, dy, dz, grip], ...]
    prompt: physical description of robot task and recovery goal.

    Adjust field names to match the deployed Cosmos action-conditioned endpoint.
    '''
    frame_b64 = image_to_base64(current_frame_path)

    return {
        "prompt": prompt,
        "input_image": frame_b64,
        "actions": action_sequence,
        "seed": seed,
        "video_params": {
            "frames_per_sec": fps,
            "frames_count": max(16, len(action_sequence)),
        },
    }

def call_action_conditioned_cosmos(payload, client=None, hosted=True):
    if client is None:
        client = CosmosClient()

    if not RUN_COSMOS_API:
        print("RUN_COSMOS_API=False. Payload preview:")
        preview = dict(payload)
        if "input_image" in preview:
            preview["input_image"] = preview["input_image"][:80] + "...<base64>"
        print(json.dumps(preview, indent=2)[:1000])
        return None

    if hosted:
        return client.infer_hosted(payload)
    return client.infer_local_nim(payload)

# 9. Bridge Gymnasium rollouts to Cosmos

The next cells define:

- how to build a Cosmos prompt from robot state and attack condition;
- how to generate candidate recovery actions;
- how to call Cosmos to predict a future;
- how to score the predicted future video;
- how to choose the best recovery action.

The scoring function is intentionally simple. Students can replace it with:
- object detector distance to red goal,
- human-labeled video quality,
- Cosmos Reason critic,
- predicted reward/value,
- or a learned success classifier.

In [ ]:
# ================================================
# 9A. Prompt builder from robot state and attack
# ================================================

def build_robot_recovery_prompt(
    condition,
    failure_mode,
    current_distance=None,
    object_displacement=None,
    candidate_action_name=None,
):
    distance_txt = "unknown" if current_distance is None else f"{current_distance:.3f} meters"
    disp_txt = "unknown" if object_displacement is None else f"{object_displacement:.3f} meters"
    action_txt = "" if candidate_action_name is None else f" Candidate recovery action: {candidate_action_name}."

    prompt = (
        "A simulated Fetch robot arm performs a pick-and-place task on a tabletop. "
        "The robot must grasp a small black cube and move it to a red goal marker. "
        f"The current condition is {condition}. "
        f"The detected failure mode is {failure_mode}. "
        f"The current distance from object to goal is {distance_txt}, and object displacement so far is {disp_txt}. "
        "Generate a physically plausible short future showing whether the robot recovers safely and moves the cube closer to the goal. "
        "The robot should avoid impossible motions, should not teleport objects, and should obey contact physics."
        f"{action_txt}"
    )
    return prompt

example_prompt = build_robot_recovery_prompt(
    condition="object_position_spoof",
    failure_mode="perception_spoof_failure",
    current_distance=0.337,
    object_displacement=0.02,
    candidate_action_name="relocalize object, then reattempt grasp"
)

print(example_prompt)

In [ ]:
# ======================================
# 9B. Candidate recovery action library
# ======================================

RECOVERY_ACTIONS = {
    "retry_grasp": {
        "description": "Move back above the cube, lower gripper, close gripper, and lift.",
        "action_sequence": [
            [0.0, 0.0, 0.3, 1.0],
            [0.0, 0.0, -0.3, 1.0],
            [0.0, 0.0, 0.0, -1.0],
            [0.0, 0.0, 0.3, -1.0],
        ],
        "best_for": ["never_grasped_object", "grasped_but_no_transport"],
    },
    "relocalize_then_grasp": {
        "description": "Pause motion, re-estimate object position, then execute a slower grasp.",
        "action_sequence": [
            [0.0, 0.0, 0.0, 1.0],
            [0.1, 0.0, 0.0, 1.0],
            [0.0, 0.0, -0.2, 1.0],
            [0.0, 0.0, 0.0, -1.0],
            [0.0, 0.0, 0.2, -1.0],
        ],
        "best_for": ["perception_spoof_failure", "sensor_dropout_failure"],
    },
    "safe_reset": {
        "description": "Stop, lift arm to safe height, and reset policy planning.",
        "action_sequence": [
            [0.0, 0.0, 0.4, 1.0],
            [0.0, 0.0, 0.2, 1.0],
            [0.0, 0.0, 0.0, 1.0],
        ],
        "best_for": ["control_channel_failure", "unknown_failure"],
    },
    "slow_transport": {
        "description": "Move object slowly toward goal to reduce overshoot under delay or clipping.",
        "action_sequence": [
            [0.05, 0.0, 0.1, -1.0],
            [0.05, 0.0, 0.0, -1.0],
            [0.05, 0.0, 0.0, -1.0],
            [0.0, 0.0, -0.1, 1.0],
        ],
        "best_for": ["moved_object_wrong_direction", "grasped_but_no_transport"],
    },
}

def candidate_recoveries_for_failure(failure_mode):
    candidates = {}
    for name, spec in RECOVERY_ACTIONS.items():
        if failure_mode in spec["best_for"]:
            candidates[name] = spec

    if not candidates:
        candidates = RECOVERY_ACTIONS

    return candidates

candidate_recoveries_for_failure("perception_spoof_failure")

In [ ]:
# ================================================
# 9C. Simple predicted-video scoring placeholder
# ================================================
# For generated videos of the Fetch scene:
# - red dot/marker roughly indicates goal;
# - dark cube roughly indicates object.
#
# This is only a starter heuristic. Replace with a detector or Cosmos Reason critic.

def score_predicted_video(video_path):
    '''
    Returns higher score for videos where the dark object appears closer to the red goal.
    This is a weak visual heuristic, intended as a template only.
    '''
    video_path = Path(video_path)

    if imageio is None or not video_path.exists():
        # Demo fallback.
        return np.random.uniform(0, 1)

    try:
        reader = imageio.get_reader(str(video_path))
        frames = []
        for i, frame in enumerate(reader):
            if i % 10 == 0:
                frames.append(frame)
        reader.close()
    except Exception:
        return np.random.uniform(0, 1)

    if not frames:
        return 0.0

    scores = []

    for frame in frames[-5:]:
        arr = np.asarray(frame)

        # Red goal marker mask.
        red_mask = (arr[:, :, 0] > 150) & (arr[:, :, 1] < 100) & (arr[:, :, 2] < 100)

        # Dark object mask.
        dark_mask = (arr[:, :, 0] < 80) & (arr[:, :, 1] < 80) & (arr[:, :, 2] < 80)

        if red_mask.sum() < 5 or dark_mask.sum() < 20:
            continue

        red_yx = np.argwhere(red_mask).mean(axis=0)
        dark_yx = np.argwhere(dark_mask).mean(axis=0)
        dist = np.linalg.norm(red_yx - dark_yx)

        # Normalize roughly by image diagonal.
        diag = np.linalg.norm(arr.shape[:2])
        scores.append(1.0 - min(1.0, dist / diag))

    if not scores:
        return 0.0

    return float(np.mean(scores))

In [ ]:
# ======================================================
# 9D. Cosmos-based recovery action selector
# ======================================================

def select_recovery_action_with_cosmos(
    current_frame_path,
    condition,
    failure_mode,
    current_distance,
    object_displacement,
    client=None,
    use_action_conditioned=False,
):
    '''
    For each candidate recovery action:
    1. Build a physical prompt.
    2. Ask Cosmos to predict future video.
    3. Score the predicted video.
    4. Return the highest-scoring recovery action.

    In demo mode, this produces placeholder scores.
    '''
    candidates = candidate_recoveries_for_failure(failure_mode)
    results = []

    for name, spec in candidates.items():
        prompt = build_robot_recovery_prompt(
            condition=condition,
            failure_mode=failure_mode,
            current_distance=current_distance,
            object_displacement=object_displacement,
            candidate_action_name=f"{name}: {spec['description']}",
        )

        output_video = VIDEO_DIR / f"cosmos_pred_{condition}_{failure_mode}_{name}.mp4"

        if use_action_conditioned and current_frame_path is not None:
            payload = build_action_conditioned_payload(
                current_frame_path=current_frame_path,
                action_sequence=spec["action_sequence"],
                prompt=prompt,
            )
            response = call_action_conditioned_cosmos(payload, client=client)
            # If response has b64_video, save it.
            if isinstance(response, dict) and "b64_video" in response:
                base64_to_file(response["b64_video"], output_video)
        else:
            # Text2World NIM-style call. This does not condition on the exact frame,
            # but still gives a concrete API integration milestone.
            cosmos_generate_video_from_prompt(prompt, output_video, client=client)

        score = score_predicted_video(output_video)

        results.append({
            "candidate": name,
            "description": spec["description"],
            "prompt": prompt,
            "predicted_video": str(output_video),
            "cosmos_score": score,
        })

    result_df = pd.DataFrame(results).sort_values("cosmos_score", ascending=False)
    best = result_df.iloc[0].to_dict()
    return best, result_df

# Demo selection.
best_action, cosmos_candidates = select_recovery_action_with_cosmos(
    current_frame_path=None,
    condition="object_position_spoof",
    failure_mode="perception_spoof_failure",
    current_distance=0.337,
    object_displacement=0.02,
)

cosmos_candidates[["candidate", "cosmos_score", "description"]]

In [ ]:
# ===================================================
# 10. Integrate recovery selector into Gymnasium loop
# ===================================================

def detect_failure_online(step_df, min_steps=20):
    '''
    Simple online detector:
    - after min_steps, if object displacement is tiny and distance remains high,
      trigger recovery.
    '''
    if len(step_df) < min_steps:
        return False, "none"

    last = step_df.iloc[-1]

    if last["success"] >= 0.5:
        return False, "success"

    if last["object_displacement"] < 0.03 and last["final_distance"] > 0.20:
        return True, "never_grasped_object"

    if last["grasp_proxy"] >= 0.5 and last["final_distance"] > 0.20:
        return True, "grasped_but_no_transport"

    if last["obs_attack"] in ["object_position_spoof", "sensor_bias"] and last["final_distance"] > 0.20:
        return True, "perception_spoof_failure"

    if last["obs_attack"] in ["sensor_dropout", "contact_dropout"] and last["final_distance"] > 0.20:
        return True, "sensor_dropout_failure"

    if last["action_attack"] in ["grip_state_falsification", "action_reversal"]:
        return True, "control_channel_failure"

    return False, "none"

def apply_recovery_action_sequence(env, action_sequence, episode_id, condition, start_t=0, save_frames=True):
    rows = []

    obs = None
    # We assume env already has current state.
    for i, action in enumerate(action_sequence):
        next_obs, reward, terminated, truncated, info = env.step(np.asarray(action, dtype=np.float32))
        success = float(info.get("is_success", 0.0)) if isinstance(info, dict) else 0.0

        rows.append({
            "episode_id": episode_id,
            "t": start_t + i,
            "condition": condition + "_cosmos_recovery",
            "obs_attack": "recovery",
            "action_attack": "recovery",
            "severity": 0.0,
            "action": action,
            "reward": float(reward),
            "terminated": bool(terminated),
            "truncated": bool(truncated),
            "success": success,
            "final_distance": get_goal_distance(next_obs),
            "object_displacement": np.nan,
            "grasp_proxy": grasp_proxy_from_obs(next_obs),
            "frame_path": save_frame(env, episode_id, start_t + i, condition + "_recovery") if save_frames else None,
        })

        if terminated or truncated:
            break

    return pd.DataFrame(rows)

def rollout_with_cosmos_recovery(
    env,
    policy,
    episode_id=0,
    condition="attacked",
    obs_attack="none",
    action_attack="none",
    severity=0.0,
    max_steps=150,
    recovery_check_after=30,
    save_frames=True,
):
    '''
    Rollout policy. If online failure detector triggers, query Cosmos selector
    and apply selected recovery action sequence.
    '''
    base_logs = []
    obs, info = env.reset()
    initial_object_pos = get_object_pos(obs)

    for t in range(max_steps):
        action, _ = policy.predict(obs, deterministic=True)
        next_obs, reward, terminated, truncated, info = env.step(action)

        frame_path = save_frame(env, episode_id, t, condition) if save_frames else None

        step_row = {
            "episode_id": episode_id,
            "t": t,
            "condition": condition,
            "obs_attack": obs_attack,
            "action_attack": action_attack,
            "severity": severity,
            "action": np.asarray(action).tolist(),
            "reward": float(reward),
            "terminated": bool(terminated),
            "truncated": bool(truncated),
            "success": float(info.get("is_success", 0.0)) if isinstance(info, dict) else 0.0,
            "final_distance": get_goal_distance(next_obs),
            "object_displacement": float(np.linalg.norm(get_object_pos(next_obs) - initial_object_pos)),
            "grasp_proxy": grasp_proxy_from_obs(next_obs),
            "frame_path": frame_path,
        }
        base_logs.append(step_row)

        current_df = pd.DataFrame(base_logs)
        trigger, failure_mode = detect_failure_online(current_df, min_steps=recovery_check_after)

        if trigger:
            best, cand_df = select_recovery_action_with_cosmos(
                current_frame_path=frame_path,
                condition=condition,
                failure_mode=failure_mode,
                current_distance=step_row["final_distance"],
                object_displacement=step_row["object_displacement"],
                use_action_conditioned=False,
            )

            action_sequence = RECOVERY_ACTIONS[best["candidate"]]["action_sequence"]
            rec_df = apply_recovery_action_sequence(
                env,
                action_sequence,
                episode_id=episode_id,
                condition=condition,
                start_t=t + 1,
                save_frames=save_frames,
            )

            out = pd.concat([current_df, rec_df], ignore_index=True)
            out["recovery_triggered"] = False
            out.loc[out.index >= len(current_df), "recovery_triggered"] = True
            out["selected_recovery"] = best["candidate"]
            return out

        obs = next_obs

        if terminated or truncated:
            break

    out = pd.DataFrame(base_logs)
    out["recovery_triggered"] = False
    out["selected_recovery"] = None
    return out

# The full function requires a live Gymnasium env. Use demo if not running.
if RUN_GYMNASIUM and gym is not None and gymnasium_robotics is not None:
    env = make_pickplace_env(obs_attack="object_position_spoof", severity=0.7)
    policy = RandomPolicy(env.action_space)
    recovery_logs = rollout_with_cosmos_recovery(
        env, policy, episode_id=999, condition="object_position_spoof"
    )
else:
    recovery_logs = logs_df.copy()
    recovery_logs["recovery_triggered"] = False
    recovery_logs["selected_recovery"] = None

recovery_logs.head()

In [ ]:
# ==========================================================
# 11. Convert rollouts to a Cosmos action-conditioned dataset
# ==========================================================
# This is a data-adapter template, not a full post-training script.
# Each row points to an input clip, action sequence, and outcome metadata.

def build_episode_video_manifest(logs_df, output_jsonl=DATA_DIR / "cosmos_action_conditioned_manifest.jsonl"):
    rows = []
    logs_df = logs_df.sort_values(["episode_id", "t"]).copy()

    for episode_id, g in logs_df.groupby("episode_id"):
        frame_paths = [p for p in g["frame_path"].dropna().tolist() if Path(p).exists()]
        if len(frame_paths) < 5:
            continue

        input_frames = frame_paths[:5]
        input_video = VIDEO_DIR / f"episode_{episode_id:04d}_input.mp4"
        make_video_from_frames(input_frames, input_video, fps=10)

        actions = g["action"].tolist()
        final = g.iloc[-1]

        row = {
            "episode_id": int(episode_id),
            "input_video": str(input_video),
            "actions": actions,
            "condition": final["condition"],
            "obs_attack": final["obs_attack"],
            "action_attack": final["action_attack"],
            "success": float(final["success"]),
            "final_distance": float(final["final_distance"]) if pd.notna(final["final_distance"]) else None,
            "prompt": build_robot_recovery_prompt(
                condition=final["condition"],
                failure_mode=label_failure_mode(final),
                current_distance=final["final_distance"],
                object_displacement=final["object_displacement"],
            )
        }
        rows.append(row)

    output_jsonl = Path(output_jsonl)
    output_jsonl.parent.mkdir(parents=True, exist_ok=True)

    with output_jsonl.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row) + "\n")

    return pd.DataFrame(rows), output_jsonl

if imageio is not None and logs_df["frame_path"].notna().any():
    manifest_df, manifest_path = build_episode_video_manifest(logs_df)
else:
    manifest_df = pd.DataFrame(columns=[
        "episode_id", "input_video", "actions", "condition",
        "obs_attack", "action_attack", "success", "final_distance", "prompt"
    ])
    manifest_path = DATA_DIR / "cosmos_action_conditioned_manifest.jsonl"

print("Manifest path:", manifest_path)
manifest_df.head()

# 12. SAC+HER training ablation template

The slides show randomized training collapsed when attacks were too aggressive.  
Do not blindly retrain another 2M-step model. First run small ablations:

- `p_clean = 0.8, 0.5, 0.2`
- attack severity small → medium → large
- attack timing before grasp / during grasp / after grasp
- per-attack curriculum instead of all attacks at once

The code below is a template for controlled training configs. It is not run by default.

In [ ]:
# ====================================
# 12A. SAC+HER training template
# ====================================

TRAINING_CONFIGS = [
    {
        "name": "clean_only",
        "clean_prob": 1.0,
        "total_timesteps": 500_000,
        "severity_start": 0.0,
        "severity_end": 0.0,
    },
    {
        "name": "curriculum_mostly_clean",
        "clean_prob": 0.8,
        "total_timesteps": 500_000,
        "severity_start": 0.1,
        "severity_end": 0.5,
    },
    {
        "name": "curriculum_balanced",
        "clean_prob": 0.5,
        "total_timesteps": 500_000,
        "severity_start": 0.1,
        "severity_end": 0.7,
    },
    {
        "name": "aggressive_randomized",
        "clean_prob": 0.2,
        "total_timesteps": 500_000,
        "severity_start": 0.2,
        "severity_end": 1.0,
    },
]

pd.DataFrame(TRAINING_CONFIGS)

In [ ]:
def train_sac_her_pickplace(config):
    '''
    Template for SAC+HER training.
    This requires a working Gymnasium Robotics + MuJoCo setup.

    Note:
    - The attack wrappers shown earlier are easiest for evaluation.
    - For training with changing random attacks every episode, make a custom env wrapper
      that samples attacks on reset().
    '''
    if SAC is None or HerReplayBuffer is None:
        raise ImportError("stable-baselines3 is required.")

    env = make_pickplace_env(render_mode=None)

    model = SAC(
        "MultiInputPolicy",
        env,
        replay_buffer_class=HerReplayBuffer,
        replay_buffer_kwargs=dict(
            n_sampled_goal=4,
            goal_selection_strategy="future",
        ),
        learning_rate=3e-4,
        buffer_size=1_000_000,
        batch_size=256,
        gamma=0.95,
        tau=0.05,
        verbose=1,
    )

    model.learn(total_timesteps=config["total_timesteps"])

    model_path = OUTPUT_DIR / f"{config['name']}_sac_her_pickplace.zip"
    model.save(model_path)
    return model, model_path

# Example:
# model, path = train_sac_her_pickplace(TRAINING_CONFIGS[0])

In [ ]:
# ===================================================
# 13. Final clean / attack / recovery result tables
# ===================================================

def make_clean_attack_recovery_table(clean_df, attack_df, recovery_df):
    def final_summary(df, label):
        if df.empty:
            return pd.DataFrame()
        _, final = summarize_episode_logs(df)
        final["stage"] = label
        return final

    all_final = pd.concat([
        final_summary(clean_df, "clean"),
        final_summary(attack_df, "attack"),
        final_summary(recovery_df, "recovery"),
    ], ignore_index=True)

    if all_final.empty:
        return pd.DataFrame(), pd.DataFrame()

    table = (
        all_final
        .groupby(["stage", "condition"], as_index=False)
        .agg(
            episodes=("episode_id", "count"),
            success_rate=("success", "mean"),
            mean_final_distance=("final_distance", "mean"),
            mean_object_displacement=("object_displacement", "mean"),
            mean_grasp_proxy=("grasp_proxy", "mean"),
        )
        .round(3)
    )

    return table, all_final

# Demo split:
clean_df = logs_df[logs_df["condition"].eq("clean")]
attack_df = logs_df[~logs_df["condition"].eq("clean")]
recovery_df = recovery_logs

final_table, all_final = make_clean_attack_recovery_table(clean_df, attack_df, recovery_df)
final_table

In [ ]:
# Save result tables.

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

benchmark_table.to_csv(OUTPUT_DIR / "tairo_attack_benchmark_table.csv", index=False)
failure_table.to_csv(OUTPUT_DIR / "tairo_failure_modes_table.csv", index=False)
cosmos_candidates.to_csv(OUTPUT_DIR / "tairo_cosmos_recovery_candidates.csv", index=False)
final_table.to_csv(OUTPUT_DIR / "tairo_clean_attack_recovery_table.csv", index=False)

print("Saved:")
for p in sorted(OUTPUT_DIR.glob("*.csv")):
    print("-", p)

In [ ]:
# Plot clean / attack / recovery comparison.

if not final_table.empty:
    plt.figure(figsize=(10, 4))
    labels = final_table["stage"] + ": " + final_table["condition"]
    plt.bar(labels, final_table["success_rate"])
    plt.ylabel("Success rate")
    plt.title("Clean vs attack vs Cosmos/world-model recovery")
    plt.xticks(rotation=45, ha="right")
    plt.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

# Week 7 checklist for the team

Use this notebook to produce these concrete artifacts:

1. **Attack benchmark table**  
   Clean SAC+HER vs each attack condition.

2. **Failure-mode table**  
   `never_grasped_object`, `grasped_but_no_transport`, `perception_spoof_failure`, etc.

3. **Cosmos integration proof**  
   One of:
   - local NIM `/v1/infer` call,
   - hosted API call using `NVIDIA_COSMOS_INVOKE_URL`,
   - local `examples/video2world.py` call using a Gymnasium frame.

4. **Cosmos recovery candidate table**  
   Candidate action, prompt, predicted video path, predicted score.

5. **Clean / attack / recovery table**  
   Success rate, final distance, object displacement, grasp proxy.

6. **Scope statement**  
   If full Cosmos Policy training is not ready, say:
   > We integrate Cosmos as a test-time world-model recovery evaluator. Full action-conditioned post-training is future work.

This is much stronger than mentioning Cosmos only in related work.